# Error Handling Patterns for Guardrails

This notebook demonstrates how to build **resilient guardrails** that handle errors gracefully.

In production systems, content filters can fail due to timeouts, malformed input, or unexpected exceptions. The key design decision is:

- **Fail-open**: Prioritize availability — if a filter crashes, allow the request through
- **Fail-closed**: Prioritize safety — if a filter crashes, block the request

This notebook also covers:
- Structured audit logging for every guardrail decision
- Graceful handling of malformed messages
- Comparison of both modes with the same inputs

## Setup

In [ ]:
# Install required packages
!pip install strands-agents strands-agents-tools --upgrade -q

In [ ]:
import logging
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Optional

# Import content filters from our shared module
import content_filters as _filters_module
ContentFilter = _filters_module.ContentFilter
FilterResult = _filters_module.FilterResult
KeywordContentFilter = _filters_module.KeywordContentFilter
RegexContentFilter = _filters_module.RegexContentFilter
Severity = _filters_module.Severity

## Audit Logging Setup

Structured audit logging captures every guardrail decision for compliance and debugging.

In [ ]:
@dataclass
class AuditEntry:
    """Structured audit log entry for a guardrail decision."""
    timestamp: str
    direction: str  # "input", "output", or "tool_call"
    filter_name: str
    action: str  # "blocked", "redacted", "warned", "passed", "error_allow", "error_block"
    content_snippet: str
    message: Optional[str] = None

    def to_log_string(self) -> str:
        snippet = self.content_snippet[:50].replace("\n", " ")
        parts = [
            f"direction={self.direction}",
            f"filter={self.filter_name}",
            f"action={self.action}",
            f'snippet="{snippet}"',
        ]
        if self.message:
            parts.append(f'message="{self.message}"')
        return " ".join(parts)


# Configure audit logging
audit_logger = logging.getLogger("guardrails.audit")
audit_logger.setLevel(logging.DEBUG)
if not audit_logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("[GUARDRAIL] %(message)s"))
    audit_logger.addHandler(handler)

## Example Filters That Fail

These filters simulate real-world failure scenarios: network errors, timeouts, and bugs.

In [ ]:
class BrokenFilter(ContentFilter):
    """A filter that always raises an exception.
    Simulates network errors, resource exhaustion, or bugs."""

    def __init__(self, name: str = "broken_filter", error_message: str = "Internal filter error"):
        super().__init__(name, Severity.BLOCK)
        self.error_message = error_message

    def evaluate(self, text: str) -> FilterResult:
        raise RuntimeError(self.error_message)


class SlowFilter(ContentFilter):
    """A filter that simulates a timeout scenario."""

    def __init__(self, name: str = "slow_filter", delay_seconds: float = 2.0):
        super().__init__(name, Severity.BLOCK)
        self.delay_seconds = delay_seconds

    def evaluate(self, text: str) -> FilterResult:
        time.sleep(self.delay_seconds)
        raise TimeoutError(f"Filter '{self.name}' timed out after {self.delay_seconds}s")

## ResilientGuardrail: Fail-Open vs Fail-Closed

This class wraps content filter execution with error handling and audit logging.

- `fail_open=True` (default): Log the error and allow the request through
- `fail_open=False`: Log the error and block the request

In [ ]:
class ResilientGuardrail:
    """A guardrail wrapper that handles filter errors gracefully."""

    def __init__(self, filters: list[ContentFilter], fail_open: bool = True, direction: str = "input"):
        self.filters = filters
        self.fail_open = fail_open
        self.direction = direction
        self.audit_log: list[AuditEntry] = []
        self._logger = logging.getLogger("guardrails.audit")

    def _create_audit_entry(self, filter_name, action, content, message=None):
        entry = AuditEntry(
            timestamp=datetime.now(timezone.utc).isoformat(),
            direction=self.direction,
            filter_name=filter_name,
            action=action,
            content_snippet=content[:50] if content else "",
            message=message,
        )
        self.audit_log.append(entry)
        return entry

    def evaluate(self, text: str) -> tuple[bool, Optional[str], Optional[str]]:
        """Evaluate text through all filters with error handling.

        Returns:
            (allowed, response_text, redacted_text)
        """
        # Handle malformed/empty input
        if text is None:
            entry = self._create_audit_entry("input_validation", "blocked", "<None>", "Received None")
            self._logger.warning(entry.to_log_string())
            return False, "Invalid input: message content is missing.", None

        if not isinstance(text, str):
            entry = self._create_audit_entry("input_validation", "blocked", str(text)[:50],
                                            f"Expected string, got {type(text).__name__}")
            self._logger.warning(entry.to_log_string())
            return False, f"Invalid input: expected text, got {type(text).__name__}.", None

        if not text.strip():
            entry = self._create_audit_entry("input_validation", "passed", "<empty>", "Empty message")
            self._logger.debug(entry.to_log_string())
            return True, None, None

        # Run each filter with error handling
        for content_filter in self.filters:
            try:
                result = content_filter.evaluate(text)

                if not result.passed:
                    if result.severity == Severity.BLOCK:
                        entry = self._create_audit_entry(content_filter.name, "blocked", text, result.message)
                        self._logger.warning(entry.to_log_string())
                        return False, f"Request blocked by '{content_filter.name}': {result.message}", None
                    elif result.severity == Severity.REDACT:
                        entry = self._create_audit_entry(content_filter.name, "redacted", text, result.message)
                        self._logger.info(entry.to_log_string())
                        text = result.redacted_text or text
                    elif result.severity == Severity.WARN:
                        entry = self._create_audit_entry(content_filter.name, "warned", text, result.message)
                        self._logger.info(entry.to_log_string())
                else:
                    entry = self._create_audit_entry(content_filter.name, "passed", text)
                    self._logger.debug(entry.to_log_string())

            except Exception as e:
                if self.fail_open:
                    entry = self._create_audit_entry(content_filter.name, "error_allow", text, f"Filter error: {e}")
                    self._logger.error(f"{entry.to_log_string()} | Allowing request (fail-open).")
                else:
                    entry = self._create_audit_entry(content_filter.name, "error_block", text, f"Filter error: {e}")
                    self._logger.error(f"{entry.to_log_string()} | Blocking request (fail-closed).")
                    return False, "Request blocked due to an internal guardrail error.", None

        return True, None, None

## Demo: Fail-Open Mode

In fail-open mode, a broken filter's error is logged but the request proceeds.

In [ ]:
# Set up filters
keyword_filter = KeywordContentFilter(name="topic_blocker", keywords=["hack", "exploit"], severity=Severity.BLOCK)
pii_filter = RegexContentFilter(
    name="pii_redactor",
    patterns=[r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"],
    severity=Severity.REDACT,
)
broken_filter = BrokenFilter(name="unstable_classifier", error_message="Connection to ML service refused")

test_inputs = [
    "What are best practices for application security?",
    "How do I hack into a system?",
    "Contact me at user@example.com for details.",
    "",
    "Normal request that will hit the broken filter",
]

print("=" * 60)
print("MODE: FAIL-OPEN (availability > safety)")
print("Filters: keyword_filter -> broken_filter -> pii_filter")
print("=" * 60)

fail_open_guardrail = ResilientGuardrail(
    filters=[keyword_filter, broken_filter, pii_filter],
    fail_open=True,
    direction="input",
)

for text in test_inputs:
    display_text = text if text else "<empty>"
    print(f"\n  Input: \"{display_text}\"")
    allowed, message, redacted = fail_open_guardrail.evaluate(text)
    print(f"  Allowed: {allowed}")
    if message:
        print(f"  Message: {message}")
    if redacted:
        print(f"  Redacted: {redacted}")

## Demo: Fail-Closed Mode

In fail-closed mode, a broken filter's error causes the request to be blocked.

In [ ]:
print("=" * 60)
print("MODE: FAIL-CLOSED (safety > availability)")
print("Filters: keyword_filter -> broken_filter -> pii_filter")
print("=" * 60)

fail_closed_guardrail = ResilientGuardrail(
    filters=[keyword_filter, broken_filter, pii_filter],
    fail_open=False,
    direction="input",
)

for text in test_inputs:
    display_text = text if text else "<empty>"
    print(f"\n  Input: \"{display_text}\"")
    allowed, message, redacted = fail_closed_guardrail.evaluate(text)
    print(f"  Allowed: {allowed}")
    if message:
        print(f"  Message: {message}")

## Demo: Malformed Message Handling

The guardrail gracefully handles unexpected input types.

In [ ]:
print("=" * 60)
print("MALFORMED MESSAGE HANDLING")
print("=" * 60)

guardrail = ResilientGuardrail(filters=[keyword_filter], fail_open=True, direction="input")

malformed_inputs = [
    (None, "None value"),
    (123, "Integer instead of string"),
    (["a", "list"], "List instead of string"),
    ("", "Empty string"),
    ("   ", "Whitespace only"),
]

for value, description in malformed_inputs:
    print(f"\n  Input ({description}): {repr(value)}")
    allowed, message, _ = guardrail.evaluate(value)
    print(f"  Allowed: {allowed}")
    if message:
        print(f"  Message: {message}")

## Comparison Table: Same Inputs, Different Modes

In [ ]:
# Suppress audit logging for cleaner output
audit_logger.setLevel(logging.CRITICAL)

comparison_open = ResilientGuardrail(
    filters=[keyword_filter, broken_filter, pii_filter], fail_open=True, direction="input"
)
comparison_closed = ResilientGuardrail(
    filters=[keyword_filter, broken_filter, pii_filter], fail_open=False, direction="input"
)

print(f"{'Input':<45} {'Fail-Open':<12} {'Fail-Closed':<12}")
print(f"{'-'*45} {'-'*12} {'-'*12}")

for text in test_inputs:
    display = (text[:42] + "...") if len(text) > 42 else text
    if not display:
        display = "<empty>"

    open_allowed, _, _ = comparison_open.evaluate(text)
    closed_allowed, _, _ = comparison_closed.evaluate(text)

    open_status = "ALLOWED" if open_allowed else "BLOCKED"
    closed_status = "ALLOWED" if closed_allowed else "BLOCKED"

    print(f"{display:<45} {open_status:<12} {closed_status:<12}")

# Restore logging
audit_logger.setLevel(logging.DEBUG)

## Audit Log Review

In [ ]:
print("AUDIT LOG SUMMARY (from fail-open guardrail)")
print("=" * 60)

for entry in fail_open_guardrail.audit_log:
    print(f"  [{entry.action.upper():12s}] filter={entry.filter_name:20s} "
          f'snippet="{entry.content_snippet[:30]}..."')

## Summary

**Key Takeaways:**

| Mode | Behavior on Error | Best For |
|------|-------------------|----------|
| **Fail-open** | Log error, allow request | User-facing apps, availability-critical systems |
| **Fail-closed** | Log error, block request | High-security environments, compliance-critical systems |

In this notebook you learned:
1. The fail-open vs fail-closed design decision
2. How to build a `ResilientGuardrail` that handles filter errors gracefully
3. Structured audit logging for compliance
4. Graceful handling of malformed inputs

**Next Steps:** See `07_testing_guardrails.ipynb` to learn how to test guardrails with example-based and property-based tests.